# 02 - Preprocessing

Notebook nay dung de lam sach du lieu cho bai toan sentiment analysis.

Quyet dinh:

- Chi dung 2 cot `Review` va `Rating`
- Chi giu `Rating` la so nguyen tu 1 den 5
- Tao nhan `sentiment` tu `Rating`
- Luu ket qua vao `data/processed/clean_reviews.csv`

## Buoc 1 - Import thu vien

In [1]:
import re
import pandas as pd
from pathlib import Path

## Buoc 2 - Khai bao duong dan

In [2]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "restaurant_reviews.csv"
PROCESSED_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "clean_reviews.csv"

print("Raw data path:", RAW_DATA_PATH)
print("Processed data path:", PROCESSED_DATA_PATH)
print("Raw file exists:", RAW_DATA_PATH.exists())

Raw data path: c:\Users\ADMIN\Downloads\Documents\DS\Project\data\raw\restaurant_reviews.csv
Processed data path: c:\Users\ADMIN\Downloads\Documents\DS\Project\data\processed\clean_reviews.csv
Raw file exists: True


## Buoc 3 - Doc du lieu raw

In [3]:
df = pd.read_csv(RAW_DATA_PATH)

print("So dong ban dau:", df.shape[0])
print("So cot ban dau:", df.shape[1])

So dong ban dau: 10000
So cot ban dau: 8


## Buoc 4 - Chi lay 2 cot can dung

`Review` la input cho model. `Rating` dung de tao nhan sentiment.

In [4]:
df_clean = df[["Review", "Rating"]].copy()

df_clean.head()

,Review,Rating
0,"The ambience was good, food was quite good . h...",5
1,Ambience is too good for a pleasant evening. S...,5
2,A must try.. great food great ambience. Thnx f...,5
3,Soumen das and Arun was a great guy. Only beca...,5
4,Food is good.we ordered Kodi drumsticks and ba...,5


## Buoc 5 - Xoa dong thieu Review hoac Rating

Dong thieu `Review` khong co input. Dong thieu `Rating` khong tao duoc label.

In [5]:
rows_before = len(df_clean)

missing_review = df_clean["Review"].isna()
missing_rating = df_clean["Rating"].isna()


df_clean = df_clean.dropna(subset=["Review", "Rating"]).copy()

rows_after = len(df_clean)
print("Thieu Review:", missing_review.sum())
print("Thieu Rating:", missing_rating.sum())
print("Thieu Review hoac Rating:", (missing_review | missing_rating).sum())
print("Thieu ca Review va Rating:", (missing_review & missing_rating).sum())
print("So dong truoc khi xoa missing:", rows_before)
print("So dong sau khi xoa missing:", rows_after)

Thieu Review: 45
Thieu Rating: 38
Thieu Review hoac Rating: 45
Thieu ca Review va Rating: 38
So dong truoc khi xoa missing: 10000
So dong sau khi xoa missing: 9955


## Buoc 6 - Chuyen Rating sang dang so

`errors="coerce"` bien gia tri khong chuyen duoc sang so thanh `NaN`, vi du `Like`.

In [6]:
df_clean["Rating"] = pd.to_numeric(df_clean["Rating"], errors="coerce")

df_clean["Rating"].value_counts(dropna=False).sort_index()

Rating
1.0    1735
1.5       9
2.0     684
2.5      19
3.0    1192
3.5      47
4.0    2373
4.5      69
5.0    3826
NaN       1
Name: count, dtype: int64

## Buoc 7 - Chi giu Rating nguyen tu 1 den 5

Chi dung cac rating ro rang: `1`, `2`, `3`, `4`, `5`.

Cac gia tri nhu `1.5`, `2.5`, `3.5`, `4.5`, `Like` se bi loai bo.

In [7]:
rows_before = len(df_clean)

valid_ratings = [1, 2, 3, 4, 5]
df_clean = df_clean[df_clean["Rating"].isin(valid_ratings)].copy()
df_clean["Rating"] = df_clean["Rating"].astype(int)

rows_after = len(df_clean)
print("So dong truoc khi loc rating:", rows_before)
print("So dong sau khi loc rating:", rows_after)
print("So dong da xoa:", rows_before - rows_after)

df_clean["Rating"].value_counts().sort_index()

So dong truoc khi loc rating: 9955
So dong sau khi loc rating: 9810
So dong da xoa: 145


Rating
1    1735
2     684
3    1192
4    2373
5    3826
Name: count, dtype: int64

## Buoc 8 - Xoa review trung lap

Neu cung mot noi dung review lap lai nhieu lan, ta chi giu lan dau tien.

In [8]:
rows_before = len(df_clean)

duplicates = df_clean["Review"].duplicated().sum()
print("So dong trung lap:", duplicates)


df_clean = df_clean.drop_duplicates(subset=["Review"]).copy()

rows_after = len(df_clean)
print("So dong truoc khi xoa duplicate:", rows_before)
print("So dong sau khi xoa duplicate:", rows_after)
print("So dong da xoa:", rows_before - rows_after)


So dong trung lap: 591
So dong truoc khi xoa duplicate: 9810
So dong sau khi xoa duplicate: 9219
So dong da xoa: 591


## Buoc 9 - Lam sach text co ban

Tao cot `review_clean` de dung cho model sau nay.

In [9]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


df_clean["review_clean"] = df_clean["Review"].apply(clean_text)

df_clean[["Review", "review_clean"]].head()

,Review,review_clean
0,"The ambience was good, food was quite good . h...",the ambience was good food was quite good had ...
1,Ambience is too good for a pleasant evening. S...,ambience is too good for a pleasant evening se...
2,A must try.. great food great ambience. Thnx f...,a must try great food great ambience thnx for ...
3,Soumen das and Arun was a great guy. Only beca...,soumen das and arun was a great guy only becau...
4,Food is good.we ordered Kodi drumsticks and ba...,food is good we ordered kodi drumsticks and ba...


## Buoc 10 - Tao nhan sentiment tu Rating

Quy tac:

- `1`, `2` -> `Negative`
- `3` -> `Neutral`
- `4`, `5` -> `Positive`

In [10]:
def rating_to_sentiment(rating):
    if rating in [1, 2]:
        return "Negative"
    if rating == 3:
        return "Neutral"
    return "Positive"


df_clean["sentiment"] = df_clean["Rating"].apply(rating_to_sentiment)

df_clean["sentiment"].value_counts()

sentiment
Positive    5671
Negative    2385
Neutral     1163
Name: count, dtype: int64

## Buoc 11 - Kiem tra du lieu sau preprocessing

In [11]:
df_clean.head()

,Review,Rating,review_clean,sentiment
0,"The ambience was good, food was quite good . h...",5,the ambience was good food was quite good had ...,Positive
1,Ambience is too good for a pleasant evening. S...,5,ambience is too good for a pleasant evening se...,Positive
2,A must try.. great food great ambience. Thnx f...,5,a must try great food great ambience thnx for ...,Positive
3,Soumen das and Arun was a great guy. Only beca...,5,soumen das and arun was a great guy only becau...,Positive
4,Food is good.we ordered Kodi drumsticks and ba...,5,food is good we ordered kodi drumsticks and ba...,Positive


In [12]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 9219 entries, 0 to 9998
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Review        9219 non-null   object
 1   Rating        9219 non-null   int32 
 2   review_clean  9219 non-null   object
 3   sentiment     9219 non-null   object
dtypes: int32(1), object(3)
memory usage: 324.1+ KB


## Buoc 12 - Luu du lieu da lam sach

File output se duoc dung cho notebook EDA va train model.

In [13]:
PROCESSED_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)

df_clean.to_csv(PROCESSED_DATA_PATH, index=False)

print("Da luu file:", PROCESSED_DATA_PATH)
print("So dong:", df_clean.shape[0])
print("So cot:", df_clean.shape[1])

Da luu file: c:\Users\ADMIN\Downloads\Documents\DS\Project\data\processed\clean_reviews.csv
So dong: 9219
So cot: 4
